# Full-text Model with Triplet Loss and Paraphrased Data

This notebook will train the detection model at the full-text embedding data for the original texts with paraphrased versions. The inputs should be six embedding files, three for original texts, and three for paraphrased texts.

# Set up

In [ ]:
import numpy as np
import pandas as pd

## Class definition

In [ ]:
import tensorflow as tf

class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(d_model, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x)
    return x


class FinetuneEncoder(tf.keras.layers.Layer):
  def __init__(self, ff_layers, dropout_rate=0.1):
    super().__init__()
    self.num_blocks = ff_layers
    self.ff_layers = [FeedForward(d_model, dropout_rate) for d_model in self.num_blocks]
    self.embedding = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))

  def call(self, x):
    for layer in self.ff_layers:
      x = layer(x)
    x = self.embedding(x)
    return x


class DistanceLayer(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
      super().__init__(**kwargs)

  def call(self, anchor, positive, negative):
      ap_distance = tf.reduce_sum(tf.square(anchor - positive), -1)
      an_distance = tf.reduce_sum(tf.square(anchor - negative), -1)
      return (ap_distance, an_distance)


class TripletModel(tf.keras.Model):
  def __init__(self, siamese_network, margin=0.5):
      super().__init__()
      self.siamese_network = siamese_network
      self.margin = margin
      self.loss_tracker = tf.keras.metrics.Mean(name="loss")

  def call(self, inputs):
      return self.siamese_network(inputs)

  def train_step(self, data):
      with tf.GradientTape() as tape:
          loss = self._compute_loss(data)
      gradients = tape.gradient(loss, self.siamese_network.trainable_weights)
      self.optimizer.apply_gradients(
          zip(gradients, self.siamese_network.trainable_weights)
      )
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def test_step(self, data):
      loss = self._compute_loss(data)
      self.loss_tracker.update_state(loss)
      return {"loss": self.loss_tracker.result()}

  def _compute_loss(self, data):
      ap_distance, an_distance = self.siamese_network(data)
      loss = ap_distance - an_distance
      loss = tf.maximum(loss + self.margin, 0.0)
      return loss

  @property
  def metrics(self):
      # We need to list our metrics here so the `reset_states()` can be
      # called automatically.
      return [self.loss_tracker]


from sklearn.metrics import f1_score

def f1(x,y,thres):
    yx = np.zeros(x.shape[0])
    yy = np.ones(y.shape[0])
    yhx = np.zeros(x.shape[0])
    yhx[x > thres] = 1
    yhy = np.zeros(y.shape[0])
    yhy[y > thres] = 1
    return f1_score(np.concatenate([yx,yy]), np.concatenate([yhx,yhy]))

In [ ]:
config = tf.compat.v1.ConfigProto()
config.gpu_options.allow_growth = True

def run_model(input_shape, n_layers, answers, gpt1, gpt2, answers_p, gpt1_p, gpt2_p, epochs=50, seeds=(1111,2222,3333,4444,5555,6666,7777,8888,9999,1010)):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []

  for seed in seeds:
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])
    ta = answers[sampling_indices < 0.8]
    tg1 = gpt1[sampling_indices < 0.8]
    tg2 = gpt2[sampling_indices < 0.8]
    tap = answers_p[sampling_indices < 0.8]
    tg1p = gpt1_p[sampling_indices < 0.8]
    tg2p = gpt2_p[sampling_indices < 0.8]
    train_answers = np.vstack([ta,ta,ta,ta,tap,tap,tap,tap])
    train_gpt1 = np.vstack([tg1,tg1,tg2,tg1p,tg1,tg1,tg2,tg1p])
    train_gpt2 = np.vstack([tg2,tg2p,tg1p,tg2p,tg2,tg2p,tg1p,tg2p])

    va = answers[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg1 = gpt1[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg2 = gpt2[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vap = answers_p[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg1p = gpt1_p[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg2p = gpt2_p[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_answers = np.vstack([va,va,va,va,vap,vap,vap,vap])
    valid_gpt1 = np.vstack([vg1,vg1,vg2,vg1p,vg1,vg1,vg2,vg1p])
    valid_gpt2 = np.vstack([vg2,vg2p,vg1p,vg2p,vg2,vg2p,vg1p,vg2p])

    test_answers = answers_p[sampling_indices >= 0.9]
    test_gpt1 = gpt1_p[sampling_indices >= 0.9]
    test_gpt2 = gpt2_p[sampling_indices >= 0.9]

    #build model
    emb_shape = input_shape
    embedding = FinetuneEncoder([emb_shape]*n_layers)
    anchor_input = tf.keras.layers.Input(name="anchor", shape=[emb_shape])
    positive_input = tf.keras.layers.Input(name="positive", shape=[emb_shape])
    negative_input = tf.keras.layers.Input(name="negative", shape=[emb_shape])
    distances = DistanceLayer()(
        embedding(anchor_input),
        embedding(positive_input),
        embedding(negative_input),
    )
    distance_network = tf.keras.Model(
        inputs=[anchor_input, positive_input, negative_input], outputs=distances
    )
    triplet_model = TripletModel(distance_network, margin=0.1)

    #training
    triplet_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), weighted_metrics=[])
    triplet_model.fit([train_gpt1, train_gpt2, train_answers],
                      epochs=epochs, batch_size=2048,
                      validation_data=[valid_gpt1, valid_gpt2, valid_answers])

    #valid accuracy
    valid_emb_answers = embedding(valid_answers)
    valid_emb_gpt1 = embedding(valid_gpt1)
    valid_emb_gpt2 = embedding(valid_gpt2)
    valid_gpt_distances = ((valid_emb_gpt1 - valid_emb_gpt2)**2).numpy().sum(axis=1)
    valid_answer_distances = ((valid_emb_gpt1 - valid_emb_answers)**2).numpy().sum(axis=1)
    valid_accuracies.append((valid_gpt_distances < valid_answer_distances).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,1,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances, valid_answer_distances, thres))

    triplet_thres = thresholds[perf.index(max(perf))]

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_emb_answers = embedding(test_answers)
    test_emb_gpt1 = embedding(test_gpt1)
    test_emb_gpt2 = embedding(test_gpt2)
    test_gpt_distances = ((test_emb_gpt1 - test_emb_gpt2)**2).numpy().sum(axis=1)
    test_answer_distances = ((test_emb_gpt1 - test_emb_answers)**2).numpy().sum(axis=1)
    test_accuracies.append((test_gpt_distances < test_answer_distances).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances, test_answer_distances, triplet_thres))
    print(valid_accuracies, test_accuracies, valid_f1s, test_f1s)
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

# Load Data

We will load embeddings for both original data and paraphrased data

In [ ]:
dataset = 'SQUAD_GPT4'

import pickle

with open(f'Data/{dataset}_answer_embs.obj', 'rb') as f:
  answers = pickle.load(f)
with open(f'Data/{dataset}_gpt1_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
with open(f'Data/{dataset}_gpt2_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)

with open(f'Data/{dataset}_paraphrased_answer_embs.obj', 'rb') as f:
  answers_p = pickle.load(f)
with open(f'Data/{dataset}_paraphrased_gpt1_embs.obj', 'rb') as f:
  gpt1_p = pickle.load(f)
with open(f'Data/{dataset}_paraphrased_gpt2_embs.obj', 'rb') as f:
  gpt2_p = pickle.load(f)

# Run Model

Similar to the other paraphrase model, you shoud set `input_shape` to the correct dimensionality of the embedding model, and tune `n_layers` a few times to see if performances increase.

In [ ]:
input_shape = 768
n_layers = 3
valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(answers,gpt1,gpt2,answers_p,gpt1_p,gpt2_p)

Quick display of performances

In [ ]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)